In [1]:
from datetime import datetime, timedelta, time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pm4py
import os

c:\Users\nikla\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Question 1: Initial Overview

### Question 2: General Visual Analytics & Exploration

### Question 3: Use Case-Specific Visual Analytics & Exploration

### (a)

In [ ]:
file_path = os.path.join(os.getcwd(), 'Event Logs', 'all.xes')
log = pm4py.read_xes(file_path)
# get cases which are rejected due to MID
mid_log = pm4py.filter_event_attribute_values(log, "code", ["MID"], level="case", retain=True)
total_cases = len(mid_log['case:concept:name'].unique())
print(f"Total cases rejected due to MID: {total_cases}\n")

# Extract and sort variants
variants_mid = pm4py.get_variants(mid_log)
variants_sorted_mid = sorted(variants_mid.items(), key=lambda x: x[1], reverse=True)

print("--- MID Trace Variants & Frequencies ---")
for i, (variant, count) in enumerate(variants_sorted_mid):
    print(f"Variant {i+1} (Frequency: {count} cases):")
    
    if isinstance(variant, str):
        print(f"  Sequence: {variant.replace(',', ' -> ')}\n")
    else:
        print(f"  Sequence: {' -> '.join(variant)}\n")

c:\Users\nikla\anaconda3\Lib\site-packages\pm4py\utils.py:1000: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1820 [00:00<?, ?it/s]

Total cases rejected due to MID: 9

--- MID Trace Variants & Frequencies ---
Variant 1 (Frequency: 4 cases):
  Sequence: Application Registered -> Retrieve Identity Document -> Retrieve Internal Log Data -> Verify ID Automatically -> Query Central Bank System -> Query Credit Bureau -> Retrieve Customer Global Risk Score -> Retrieve Financial Status -> Create Assessment Report -> Verify ID Manually -> Reject Application -> Send Rejection Notification

Variant 2 (Frequency: 1 cases):
  Sequence: Application Registered -> Retrieve Identity Document -> Verify ID Automatically -> Retrieve Internal Log Data -> Query Central Bank System -> Query Credit Bureau -> Retrieve Financial Status -> Retrieve Customer Global Risk Score -> Create Assessment Report -> Verify ID Manually -> Reject Application -> Send Rejection Notification

Variant 3 (Frequency: 1 cases):
  Sequence: Application Registered -> Retrieve Identity Document -> Retrieve Internal Log Data -> Verify ID Automatically -> Reject App

### Answser for a:
In Variants 3 and 4 (both 1 case), even after the initial application is rejected, the event is followed by unnecessary activities, such as querying the central bank and credit bureau. This is not very efficient as it wastes resources and time.

### (b)

In [ ]:

#load lcs sublog
file_path_lcs = os.path.join(os.getcwd(), 'Event Logs','lcs-rejected-customers.xes')
raw_data = pm4py.read_xes(file_path_lcs)
log_lcs = pm4py.convert_to_event_log(raw_data)

cases_after_decide = []
cases_without_decide = []

#loop through the logs 

for trace in log_lcs:
    case_id = trace.attributes.get('concept:name', 'Unknown')
    
    # store current activities and codes
    activities = [event['concept:name'] for event in trace]
    codes = [event.get('code', None) for event in trace]
    
    # check if this case was actually rejected with LCS
    if 'LCS' in codes:
        score_info = "Not Found"
        
        # check for all events in the case
        for event in trace:
            #search for riskscore
            if event['concept:name'] == 'Retrieve Customer Global Risk Score':
                # only get important columns
                score_info = {k: v for k, v in event.items() if k not in ['concept:name', 'time:timestamp', 'code'] and str(v) != 'nan'}
                break # Stop searching this trace once we find it
                
        # sort if decide loan application is included
        if 'Decide Loan Application' in activities:
            cases_after_decide.append((case_id, score_info))
        else:
            cases_without_decide.append((case_id, score_info))

#print the required 2 cases for each prompt
print("--- 2 Cases Rejected (LCS) AFTER 'Decide Loan Application' ---")
for case_id, score in cases_after_decide[:2]:
    print(f"Case ID: {case_id} | Risk Score Data: {score}")

print("\n--- 2 Cases Rejected (LCS) BEFORE 'Decide Loan Application' ---")
for case_id, score in cases_without_decide[:2]:
    print(f"Case ID: {case_id} | Risk Score Data: {score}")


### (c)

In [ ]:
# define target activities
target_activities = [
    'Reject Application', 
    'Decide Loan Application', 
    'Send to Collections'
]
# project the logs, only keep the target events
projected_log = pm4py.filter_event_attribute_values(
    log_lcs, 
    "concept:name", 
    target_activities, 
    level="event", 
    retain=True
)

#get the different variants 
variants_proj = pm4py.get_variants(projected_log)

# order variants
variants_proj_sorted = sorted(variants_proj.items(), key=lambda x: len(x[1]), reverse=True)


print("--- Projected LCS Trace Variants & Frequencies ---")
for i, (variant, traces) in enumerate(variants_proj_sorted):
    print(f"Variant {i+1} (Frequency: {len(traces)} cases):")
    
    # Format the output beautifully with arrows
    if isinstance(variant, str):
        print(f"  Sequence: {variant.replace(',', ' -> ')}\n")
    else:
        print(f"  Sequence: {' -> '.join(variant)}\n")

### Question 4: Process Discovery

### Question 5: Conformance Checking